# Inspect the database — high-level observation

Read-only overview of the Postgres store: what each pipeline stage wrote, where it lives, and how much is there.
Nothing here writes — every query is `SELECT` scoped to one `(tenant_id, project_id)`.


In [ ]:
from pathlib import Path
import json
from sqlalchemy import text
from potency_kg.db import create_engine_from_env

nb_dir = Path.cwd().resolve()
package_root = nb_dir.parent if nb_dir.name == "notebooks" else nb_dir
config_path = package_root / "configs/casgevy-potency.json"
scope = json.loads(config_path.read_text())["scope"]
TENANT, PROJECT = scope["tenant_id"], scope["project_id"]
engine = create_engine_from_env()
print(f"scope: {TENANT}/{PROJECT}")
print(f"config: {config_path}")


## Where is what

Postgres 17 (+ `pgvector`, `age`) — host `db:5432` inside compose, `localhost:5432` from host. Connect string comes from `POSTGRES_USER/PASSWORD/HOST/PORT/DB` (see `lib/potency_kg/db.py`, `etc/.env`).
Schema is owned by Alembic: `migrations/versions/0001..0005`.

| Pipeline step | Tables | What they mean |
|---|---|---|
| Import (structural) | `import_run`, `source_document`, `source_head`, `source_passage`, `import_error` | Raw evidence: runs, documents (one row per source_version), active head pointer, 791 passages, failures |
| Aliases (catalog) | `aliases`, `alias_evidence`, `alias_discovery_run` | `approved` = usable for Jev/extraction; `pending` = proposed, needs review; evidence = why (static seed / deterministic rule / llm / extraction quote) |
| Embeddings + index | `embedding_run`, `passage_embedding`, `index_snapshot`, `index_member`, `current_index` | One row per passage attempt (`succeeded`/`failed`); index = frozen set of embeddings; `current_index` = active one |
| Jev (selection) | `jev_response`, `jev_selection` | Per-passage model scores, then selected/not threshold decision |
| Extraction + materialize | `llm_extraction_run`, `validation_failure`, `passage_processing`, `materialization_run`, `entity_mention`, `redaction_mention`, `assertion` | Structured claims per passage; `passage_processing.state` = funnel position (`not_run` → `structural_only` → `selected` → `extracted`); assertions = evidence-bearing claims |
| Graph (KG) | `kg_node`, `kg_edge`, `kg_node_source_ref`, `kg_edge_source_ref`, `kg_node_mention_ref`, `graph_scope`, `graph_refresh_run` | Relational projection: `structural` = only SourceDocument/SourcePassage/`contains_passage`; `semantic` adds Product/Assay/Assertion etc.; `graph_scope` = active version, refresh runs = shadow-build history |

Every table is scoped by `(tenant_id, project_id)` — the isolation-fixture scopes never mix.


In [ ]:
PARAMS = {"tenant": TENANT, "project": PROJECT}

def count(table):
    with engine.connect() as c:
        return c.execute(text(f"SELECT count(*) FROM {table} WHERE tenant_id=:tenant AND project_id=:project"), PARAMS).scalar_one()

def breakdown(table, col):
    with engine.connect() as c:
        try:
            rows = c.execute(text(f"SELECT {col}, count(*) FROM {table} WHERE tenant_id=:tenant AND project_id=:project GROUP BY 1 ORDER BY 2 DESC"), PARAMS).all()
            return [(r[0], r[1]) for r in rows]
        except Exception as e:
            return [(f"<no {col}: {type(e).__name__}>", 0)]

def report(title, tables):
    print(f"--- {title} ---")
    for t in tables:
        try:
            print(f"{t:28s} {count(t):6d}")
        except Exception as e:
            print(f"{t:28s} ERROR {type(e).__name__}: {e}")
    print()

def sample(table, cols="*", limit=3):
    with engine.connect() as c:
        rows = c.execute(text(f"SELECT {cols} FROM {table} WHERE tenant_id=:tenant AND project_id=:project LIMIT {int(limit)}"), PARAMS).mappings().all()
        for r in rows:
            print(dict(r))
        if not rows:
            print(f"(no rows in {table} for this scope)")


## 1. Evidence (import) — documents and passages
`source_head` = active version per `source_id`. `source_document.expected_passage_count` vs actual `source_passage` rows exposes truncation. `import_error` must be empty on a clean run.

In [ ]:
report("evidence", ["import_run", "source_document", "source_head", "source_passage", "import_error"])
print("source_document.status:", breakdown("source_document", "status"))
print("source_document.authority:", breakdown("source_document", "authority"))
print("import_run.status:", breakdown("import_run", "status"))

with engine.connect() as c:
    rows = c.execute(text("""
      SELECT d.source_id, d.expected_passage_count, count(p.passage_id) AS actual
      FROM source_document d LEFT JOIN source_passage p USING (tenant_id, project_id, source_id, source_version)
      WHERE d.tenant_id=:tenant AND d.project_id=:project
      GROUP BY 1, 2 ORDER BY 1"""), PARAMS).all()
    print(f"documents: {len(rows)}, passages: {sum(r[2] for r in rows)}")
    for r in rows[:10]:
        flag = "" if r[1] == r[2] else "  <-- MISMATCH"
        print(f"  {r[0]:40s} expected={r[1]:4d} actual={r[2]:4d}{flag}")


## 2. Aliases — the catalog Jev/extraction actually uses
`aliases.status`: `approved` (usable) vs `pending` (proposed, needs `review-alias`) vs `rejected`. `alias_evidence.source_type`: `static_config` (seed file) / `deterministic_rule` / `llm_mined` / `extraction`. Only approved aliases land in `alias_catalog_version` stamped on `jev_response` / `llm_extraction_run`.

In [ ]:
report("aliases", ["aliases", "alias_evidence", "alias_discovery_run"])
print("aliases.status:", breakdown("aliases", "status"))
print("aliases.entity_type:", breakdown("aliases", "entity_type"))
print("alias_evidence.source_type:", breakdown("alias_evidence", "source_type"))
print("alias_discovery_run: (method, status)")
with engine.connect() as c:
    for r in c.execute(text("SELECT method, status, count(*) FROM alias_discovery_run WHERE tenant_id=:tenant AND project_id=:project GROUP BY 1,2"), PARAMS).all():
        print(" ", tuple(r))


## 3. Semantic funnel — embeddings → Jev → extraction → materialization
`passage_processing.state` is the funnel: `not_run` → `structural_only` → `selected` (Jev) → `extracted`, plus `failed`. Counts here must reconcile: `selected` ≈ `jev_selection` selected rows, `extracted` ≈ succeeded `llm_extraction_run`.

In [ ]:
report("semantic", ["embedding_run", "passage_embedding", "index_snapshot", "index_member",
                  "jev_response", "jev_selection", "llm_extraction_run", "validation_failure",
                  "passage_processing", "materialization_run", "entity_mention", "redaction_mention", "assertion"])
print("passage_processing.state:", breakdown("passage_processing", "state"))
print("embedding_run.status:", breakdown("embedding_run", "status"))
print("jev_response.status:", breakdown("jev_response", "status"))
print("llm_extraction_run.status:", breakdown("llm_extraction_run", "status"))
print("materialization_run.status:", breakdown("materialization_run", "status"))
with engine.connect() as c:
    print("current_index:", c.execute(text("SELECT index_version, embedding_version FROM current_index WHERE tenant_id=:tenant AND project_id=:project"), PARAMS).mappings().all())


## 4. Knowledge graph — relational projection + AGE shadow
`kg_node.node_type` / `kg_edge.edge_type` tell structural (`SourceDocument`/`SourcePassage`/`contains_passage`, `system_generated`) apart from semantic evidence edges (`is_evidence=true` → has `assertion_id`). `graph_scope` = currently served version; `graph_refresh_run` = build history (`structural` vs `semantic`). The AGE graph itself (`kg_<hex>_<hex>`) mirrors these tables — inspect via `refresh_graph_stage` / `results/kg` export, not raw AGE cypher.

In [ ]:
report("graph", ["kg_node", "kg_edge", "kg_node_source_ref", "kg_edge_source_ref",
               "kg_node_mention_ref", "graph_scope", "graph_refresh_run"])
print("kg_node.node_type:", breakdown("kg_node", "node_type"))
print("kg_edge.edge_type:", breakdown("kg_edge", "edge_type"))
with engine.connect() as c:
    print("evidence edges:", c.execute(text("SELECT count(*) FROM kg_edge WHERE tenant_id=:tenant AND project_id=:project AND is_evidence"), PARAMS).scalar_one())
    print("graph_scope:", c.execute(text("SELECT active_graph_name, graph_version, index_version, embedding_version FROM graph_scope WHERE tenant_id=:tenant AND project_id=:project"), PARAMS).mappings().all())
    print("graph_refresh_run (mode, status):", c.execute(text("SELECT mode, status, count(*) FROM graph_refresh_run WHERE tenant_id=:tenant AND project_id=:project GROUP BY 1,2"), PARAMS).all())


## 5. Pipeline rollup — one line per stage
Empty downstream tables after a structural-only run are expected (no model gate → no embeddings/Jev/extraction/semantic graph). A healthy full run: `source_passage` = 791, `import_error` = 0, approved aliases > 0, `passage_processing.extracted` > 0, semantic `graph_scope` bound to an `index_version`.

In [ ]:
stages = {
    "import": ["source_document", "source_passage"],
    "aliases": ["aliases", "alias_evidence"],
    "embeddings": ["embedding_run", "index_snapshot"],
    "jev": ["jev_response", "jev_selection"],
    "extraction": ["llm_extraction_run", "assertion", "entity_mention"],
    "graph": ["kg_node", "kg_edge", "graph_refresh_run"],
}
for stage, tables in stages.items():
    print(f"{stage:12s}", {t: count(t) for t in tables})

# Samples for sanity: one document, one passage, one KG node, one assertion
print()
sample("source_document", "source_id, source_version, title, status, authority", 2)
sample("source_passage", "source_id, passage_id, left(text, 120) AS text_head", 1)
sample("kg_node", "node_id, node_type", 2)
sample("assertion", "assertion_id, passage_id, left(claim, 120) AS claim_head", 1)
